# Resolución de Sudokus — Red de grafos (GNN / Recurrent Relational Network)

A diferencia del MLP denso, aquí la **arquitectura refleja las reglas del sudoku**:

- **Nodos:** las 81 celdas.
- **Aristas:** cada celda está conectada a las **20** con las que comparte fila,
  columna o bloque 3×3 (sus *peers*). El grafo es el mismo para todos los sudokus.
- **Mensajes:** en cada paso, cada nodo recibe información de sus vecinos y actualiza
  su estado con una GRU. Tras varios pasos, "razona" sobre las restricciones.
- **Salida:** por cada nodo, 9 logits (dígitos 1–9).

Es la idea de las *Recurrent Relational Networks* (Palm et al., 2018), implementada
con torch puro: como las aristas son fijas, precomputamos los vecinos y hacemos el
paso de mensajes con `gather`, sin librerías de grafos.

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

ROOT = Path.cwd().parent
CSV = ROOT / "data" / "sudoku.csv"
device = "mps" if torch.backends.mps.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

device: mps


## 1. Muestra aleatoria de 1.000.000

Igual que en el modelo denso: marcamos 1M filas al azar y leemos el CSV por chunks.

In [2]:
N_TOTAL, N_SAMPLE = 9_000_000, 1_000_000
rng = np.random.default_rng(42)
keep = np.zeros(N_TOTAL, dtype=bool)
keep[rng.choice(N_TOTAL, N_SAMPLE, replace=False)] = True

puzzles, solutions = [], []
start, t0 = 0, time.time()
for chunk in pd.read_csv(CSV, chunksize=1_000_000, dtype=str):
    n = len(chunk)
    sub = chunk[keep[start:start + n]]
    puzzles.append(sub["puzzle"].to_numpy())
    solutions.append(sub["solution"].to_numpy())
    start += n
puzzles   = np.concatenate(puzzles)
solutions = np.concatenate(solutions)
print(f"muestreados {len(puzzles):,} en {time.time()-t0:.1f}s")

muestreados 1,000,000 en 8.8s


## 2. Codificar

La GNN usa el dígito **0–9 tal cual** (un `Embedding` aprende un vector por dígito;
el 0 = celda vacía tiene su propio vector), así que la entrada son enteros, no `/9`.
La salida son clases 0–8 (dígito − 1).

In [3]:
def a_rejilla(strs):
    a = np.frombuffer("".join(strs.tolist()).encode("ascii"), dtype=np.uint8) - ord("0")
    return a.reshape(-1, 81)

P = a_rejilla(puzzles)      # (N,81) 0-9
S = a_rejilla(solutions)    # (N,81) 1-9

X = torch.from_numpy(P.astype(np.int64))        # enteros (para el Embedding)
Y = torch.from_numpy((S - 1).astype(np.int64))  # clases 0-8

perm = torch.randperm(len(X))
X, Y = X[perm], Y[perm]

# Entrenamiento serio: mas sudokos = mejor. Sube N_TRAIN si tienes tiempo/paciencia
# (cada epoca tarda ~ proporcional a N_TRAIN). Con 200k ya se ve subir bien.
N_TRAIN, N_VAL = 200_000, 10_000
Xtr, Ytr = X[:N_TRAIN], Y[:N_TRAIN]
Xva, Yva = X[N_TRAIN:N_TRAIN + N_VAL], Y[N_TRAIN:N_TRAIN + N_VAL]
train_dl = DataLoader(TensorDataset(Xtr, Ytr), batch_size=256, shuffle=True)
print("train:", Xtr.shape, "| val:", Xva.shape)

train: torch.Size([200000, 81]) | val: torch.Size([10000, 81])


## 3. Vecinos del grafo

Para cada celda, las 20 *peers* (misma fila, columna o bloque). Tensor fijo `(81, 20)`.

In [4]:
def construir_vecinos():
    vecinos = []
    for r in range(9):
        for c in range(9):
            peers = set()
            for cc in range(9): peers.add(r * 9 + cc)         # fila
            for rr in range(9): peers.add(rr * 9 + c)         # columna
            br, bc = 3 * (r // 3), 3 * (c // 3)
            for rr in range(br, br + 3):                      # bloque 3x3
                for cc in range(bc, bc + 3):
                    peers.add(rr * 9 + cc)
            peers.discard(r * 9 + c)
            vecinos.append(sorted(peers))
    return torch.tensor(vecinos, dtype=torch.long)            # (81, 20)

VECINOS = construir_vecinos()
print("vecinos por celda:", VECINOS.shape[1])

vecinos por celda: 20


## 4. Modelo (paso de mensajes recurrente)

En cada uno de los `steps`:
1. cada nodo mira el estado de sus 20 vecinos y calcula un **mensaje** (MLP),
2. **suma** los mensajes recibidos,
3. **actualiza** su estado con una `GRUCell` (recordando también su dígito inicial).

Se produce una predicción en cada paso; la pérdida suma todos los pasos para forzar
que el razonamiento converja.

In [5]:
class SudokuGNN(nn.Module):
    def __init__(self, vecinos, H=96, steps=8):
        super().__init__()
        self.register_buffer("nb", vecinos)      # (81,20)
        self.H, self.steps = H, steps
        self.embed = nn.Embedding(10, H)         # digito 0-9 -> vector H
        self.msg = nn.Sequential(nn.Linear(2 * H, H), nn.ReLU(),
                                 nn.Linear(H, H), nn.ReLU())
        self.gru = nn.GRUCell(2 * H, H)          # entrada: [mensaje, embedding inicial]
        self.out = nn.Linear(H, 9)

    def forward(self, x):                         # x: (B,81) enteros
        B = x.size(0)
        xe = self.embed(x)                        # (B,81,H) rasgo fijo de entrada
        h = xe.clone()                            # estado inicial de cada nodo
        salidas = []
        for _ in range(self.steps):
            hj = h[:, self.nb]                                  # (B,81,20,H) vecinos
            hi = h.unsqueeze(2).expand(-1, -1, self.nb.size(1), -1)
            m = self.msg(torch.cat([hi, hj], -1)).sum(2)        # (B,81,H) mensajes sumados
            gin = torch.cat([m, xe], -1).reshape(B * 81, -1)
            h = self.gru(gin, h.reshape(B * 81, self.H)).reshape(B, 81, self.H)
            salidas.append(self.out(h))           # (B,81,9) prediccion de este paso
        return salidas

model = SudokuGNN(VECINOS).to(device)
print(model)

SudokuGNN(
  (embed): Embedding(10, 96)
  (msg): Sequential(
    (0): Linear(in_features=192, out_features=96, bias=True)
    (1): ReLU()
    (2): Linear(in_features=96, out_features=96, bias=True)
    (3): ReLU()
  )
  (gru): GRUCell(192, 96)
  (out): Linear(in_features=96, out_features=9, bias=True)
)


## 5. Entrenamiento

In [ ]:
from tqdm.auto import tqdm

EPOCHS = 30
opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
sched = torch.optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5)  # baja el lr cada 10
crit  = nn.CrossEntropyLoss()
dst   = ROOT / "models" / "sudoku_solver_gnn.pt"     # SOLO los pesos del mejor (para inferencia)
ckpt  = ROOT / "models" / "sudoku_solver_gnn.ckpt"   # estado completo para reanudar


@torch.no_grad()
def evaluar(Xv, Yv, bs=512):
    model.eval()
    cell_ok = board_ok = 0
    for i in range(0, len(Xv), bs):
        pred = model(Xv[i:i+bs].to(device))[-1].argmax(-1).cpu()   # ultimo paso
        eq = pred == Yv[i:i+bs]
        cell_ok  += eq.sum().item()
        board_ok += eq.all(1).sum().item()
    return cell_ok / (len(Xv) * 81), board_ok / len(Xv)


# --- Reanudar / warm-start ------------------------------------------------------
# Prioridad:
#   1) .ckpt  -> reanuda TODO (modelo, optimizador, scheduler, epoca, mejor marca).
#   2) .pt    -> warm-start: solo los pesos del mejor de un run anterior (sin .ckpt).
#   3) nada   -> desde cero.
# Puedes cortar el entrenamiento cuando quieras: al re-ejecutar esta celda retoma
# desde el ultimo checkpoint. Para empezar limpio, borra el .ckpt (y el .pt).
start_ep, mejor = 1, 0.0
if ckpt.exists():
    estado = torch.load(ckpt, map_location=device)
    model.load_state_dict(estado["model"])
    opt.load_state_dict(estado["opt"])
    sched.load_state_dict(estado["sched"])
    start_ep = estado["epoch"] + 1
    mejor    = estado["mejor"]
    print(f"reanudando desde checkpoint: epoca {start_ep}/{EPOCHS}, mejor acc_tablero {mejor:.3f}")
elif dst.exists():
    # warm-start: no hay checkpoint completo pero si los pesos del mejor modelo
    # de un entrenamiento anterior. Los cargamos para no empezar de cero (el
    # optimizador/scheduler arrancan limpios). Evaluamos para fijar 'mejor' y no
    # sobrescribir un buen .pt con una epoca peor.
    model.load_state_dict(torch.load(dst, map_location=device))
    _, mejor = evaluar(Xva, Yva)
    print(f"warm-start desde {dst.name}: acc_tablero actual {mejor:.3f} (optimizador desde cero)")
else:
    print("sin checkpoint ni pesos previos: entrenando desde cero")

# --- Bucle de entrenamiento -----------------------------------------------------
for ep in range(start_ep, EPOCHS + 1):
    model.train()
    # barra por LOTE: ves el avance dentro de cada epoca en tiempo real
    barra = tqdm(train_dl, desc=f"epoch {ep}/{EPOCHS}", leave=False)
    for xb, yb in barra:
        xb, yb = xb.to(device), yb.to(device)
        salidas = model(xb)
        loss = sum(crit(s.reshape(-1, 9), yb.reshape(-1)) for s in salidas) / len(salidas)
        opt.zero_grad(); loss.backward(); opt.step()
        barra.set_postfix(loss=f"{loss.item():.3f}")
    sched.step()
    acc_cel, acc_tab = evaluar(Xva, Yva)

    marca = ""
    if acc_tab > mejor:                      # guarda los pesos del mejor (para inferencia)
        mejor = acc_tab
        torch.save(model.state_dict(), dst)
        marca = "  <- guardado (mejor)"

    # checkpoint de reanudacion: se guarda SIEMPRE, al final de cada epoca
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(),
                "sched": sched.state_dict(), "epoch": ep, "mejor": mejor}, ckpt)

    print(f"epoch {ep}/{EPOCHS}  acc_celda {acc_cel:.3f}  acc_tablero {acc_tab:.3f}{marca}")

print(f"\nmejor acc_tablero: {mejor:.3f}  ->  {dst}")

warm-start desde sudoku_solver_gnn.pt: acc_tablero actual 0.891 (optimizador desde cero)


epoch 1/30:   0%|          | 0/782 [00:00<?, ?it/s]

epoch 1/30  acc_celda 0.991  acc_tablero 0.881


epoch 2/30:   0%|          | 0/782 [00:00<?, ?it/s]

epoch 2/30  acc_celda 0.992  acc_tablero 0.883


...

epoch 6/30:   0%|          | 0/782 [00:00<?, ?it/s]

epoch 6/30  acc_celda 0.993  acc_tablero 0.896  <- guardado (mejor)


epoch 7/30:   0%|          | 0/782 [00:00<?, ?it/s]

epoch 7/30  acc_celda 0.993  acc_tablero 0.898  <- guardado (mejor)


epoch 8/30:   0%|          | 0/782 [00:00<?, ?it/s]

## 6. Guardar y comparar con el solver (oráculo)

In [ ]:
import sys
sys.path.append(str(ROOT / "src"))
import solver

# usa el MEJOR modelo guardado durante el entrenamiento (no el de la ultima epoca)
model.load_state_dict(torch.load(dst, map_location=device))
model.eval()

@torch.no_grad()
def resolver_con_gnn(puzzle81):
    x = torch.tensor(puzzle81, dtype=torch.long).view(1, 81)
    pred = model(x.to(device))[-1].argmax(-1).cpu().numpy().reshape(9, 9) + 1
    return pred.tolist()

ej = P[N_TRAIN]                          # un puzzle no visto en entrenamiento
tablero = ej.reshape(9, 9).tolist()
gnn = resolver_con_gnn(ej)
bt  = solver.solve(tablero)

print("Puzzle:");           solver.print_board(tablero)
print("\nGNN:");           solver.print_board(gnn)
print("\nSolver (backtracking):"); solver.print_board(bt)
print("\nMetricas GNN vs solver:", solver.evaluar_solucion(gnn, bt))